# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tkg-create/FlyRank-ML-Track/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

REPO_URL = "https://github.com/tkg-create/FlyRank-ML-Track.git"
REPO_DIR = "FlyRank-ML-Track"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())
assert os.path.isfile("work/scripts/01_load_and_score.py"), "01_load_and_score.py not found — did the clone work?"

Cloning into 'FlyRank-ML-Track'...
remote: Enumerating objects: 408, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 408 (delta 37), reused 24 (delta 19), pack-reused 336 (from 2)
Receiving objects: 100% (408/408), 2.06 MiB | 5.51 MiB/s, done.
Resolving deltas: 100% (223/223), done.
Working dir: /content/FlyRank-ML-Track


## 1. Question

*The research question and the decision it supports.*

Organizations that manage content across many sites face the same mismatch: the portfolio of pages is far larger than the capacity to review it by hand. Editorial attention is scarce, so it has to be spent on the pages where it matters most. Given a fixed review capacity, which declining pages should be reviewed first, and what should a reviewer actually do with each one?

This work tests whether a model-based ranking answers that question better than a hand-built rule, for a portfolio of several dozen client sites and hundreds of thousands of pages. Review capacity is set at 50 pages per month (`REVIEW_CAPACITY_K`, defined below), an assumed budget, chosen because it's also where a model-informed ranking held its advantage over the rule most consistently under grouped cross-validation, winning in 5 of 5 folds (see Methodology and Results).

What follows validates that approach against one month of historical data under honest cross-validation. It doesn't deliver a tool that scores new pages every month on its own yet, see Limitations for what that would still take.

In [2]:
"""
Assumed monthly review capacity — chosen because grouped cross-validation found
this is where a model-informed ranking's edge over the rule is most consistent (5/5 folds),
not just where it looks best on average. See Methodology and Results.
"""
REVIEW_CAPACITY_K = 50

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Tables:** Each row represents one page, on one date, for one client. The table is `fact_content_daily_performance`, filtered to the `month=2026-03` partition of the internship warehouse. That month was chosen deliberately. Earlier months are missing analytics history for many clients, and later months overlap a separate 90-day table's window in a way that risks the label and features bleeding into each other. A month in the middle of the panel avoids both problems.

**Date windows:** The single month is split into two windows. The first half of the month versus the second half, March 1 to 15 against March 16 to 31, defines the decline label. Week 1 versus week 2 of the first half, March 1 to 8 against March 9 to 15, defines the position-trend feature. The feature window sits strictly before the label window, so no feature can see into the period it's predicting.

**Excluded, and why:** The raw first-half and second-half impression counts that the label is built from are excluded from every feature set. Adding one of them back in as a diagnostic pushed a model's ROC AUC from 0.564 to 0.991, which meant the model was reconstructing the label algebraically rather than learning anything real. A content-metadata table was excluded because it turned out to be a current-state snapshot rather than a point-in-time history. Joining it onto daily rows produced impossible negative day counts. Rows without a confirmed analytics coverage flag were excluded, since those rows measure "not tracked" rather than a genuine zero. A session-rate engagement signal was excluded as well, after earlier analysis found it confounded and pointing in the wrong direction, and has been left out of every model built since.

**Public-safe:** Every identifier used for joining or grouping is a hashed ID and is never used as a model feature. No client name, URL, or search query text appears anywhere in this pipeline's outputs.

In [3]:
import subprocess
import sys
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face READ token: ")

process = subprocess.Popen(
    [sys.executable, "-u", "work/scripts/01_load_and_score.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
process.wait()
assert process.returncode == 0, f"01_load_and_score.py failed with exit code {process.returncode}"

Paste your Hugging Face READ token: ··········
Loading and building features from the warehouse...
  [1/6] Querying label (impressions first half vs second half)...
        -> 151,981 rows
  [2/6] Querying full-month position/click signal (baseline rule)...
        -> 175,304 rows
  [3/6] Querying first-half-only features (model training data)...
        -> 150,675 rows
  [4/6] Querying leakage-safe position trend (week 1 vs week 2)...
        -> 119,176 rows
  [5/6] Querying client map (grouping key only, never a feature)...
        -> 331,437 rows
  [6/6] Merging into model_df...
model_df: (150675, 16), base rate: 0.438
Running 5-fold GroupKFold OOF scoring (random_state=42)...
  Fold 1/5: fitting on 120,583 rows, scoring 30,092...
  Fold 1/5: done
  Fold 2/5: fitting on 120,589 rows, scoring 30,086...
  Fold 2/5: done
  Fold 3/5: fitting on 120,351 rows, scoring 30,324...
  Fold 3/5: done
  Fold 4/5: fitting on 120,589 rows, scoring 30,086...
  Fold 4/5: done
  Fold 5/5: fitting on 

In [4]:
import pandas as pd

model_df = pd.read_csv("work/data/processed/w07_scored_population.csv")
print(f"Rows: {len(model_df):,}")
print(f"Base rate (share labeled declining): {model_df['is_declining_proxy'].mean():.3f}")
model_df[["avg_position_fh", "log_impressions_fh", "log_clicks_fh", "ctr_fh", "position_change", "has_position_trend"]].describe()

Rows: 150,675
Base rate (share labeled declining): 0.438


,avg_position_fh,log_impressions_fh,log_clicks_fh,ctr_fh,position_change,has_position_trend
count,150675.000000,150675.000000,150675.000000,150675.000000,150675.000000,150675.000000
mean,16.546579,4.630053,0.526925,0.004285,0.618596,0.790947
std,18.215346,2.287959,0.901977,0.034230,9.208292,0.406633
min,0.040763,0.693147,0.000000,0.000000,-213.750000,0.000000
25%,5.186723,2.833213,0.000000,0.000000,-0.950392,1.000000
50%,8.756892,4.709530,0.000000,0.000000,0.000000,1.000000
75%,21.131311,6.410175,0.693147,0.002023,1.666203,1.000000
max,310.000000,11.992731,7.781556,1.000000,153.750000,1.000000


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Assumptions:** Pages need at least 10 impressions across the month to be trusted for position and click-through numbers. Pages below that bar are excluded rather than treated as having a real value of zero. The decline label is a proxy, not proof of real decline. Comparing impressions across two halves of one month captures a within-month direction, not a causal or long-term trend. Validation is grouped by client, on the assumption that a model shouldn't be tested on a client whose other pages it trained on. That's a reasonable default, but not the only possible choice; grouping by content topic or launch date was not tested.

**Model and features:** The model is a Random Forest with 300 trees, a max depth of 6, and a minimum of 20 samples per leaf. It trains on six features: average position, log impressions, log clicks, click-through rate, position change, and whether a position trend exists for the page at all. A seventh candidate feature was built during earlier validation work, a zero-click flag similar to the one used in the baseline rule. It was left out of the final feature set on purpose, since it only ever backed an earlier diagnostic comparison. As a result the model sees click-through problems only indirectly, through CTR and click volume, rather than as an explicit flag the way the baseline rule does.

**Label:** A page is labeled declining by comparing its search impressions in the first half of the month to the second half. The raw impression counts from either half are never allowed as features. This was confirmed necessary rather than just cautious. Adding one of those counts back in as a diagnostic pushed a model from barely above chance to near perfect. That jump wasn't real learning. It was the model reconstructing the label algebraically from a column that was almost the label itself.

**Baseline:** The baseline is a transparent rule with two parts. One flag catches a page holding a top-10 search position with zero clicks. The other catches a page whose position got worse from the first week of the month to the second. The zero-click flag counts for more than the position flag when they're combined into a single score. Every input is knowable at the decision moment, so nothing here requires a trained model.

**Validation design:** Validation uses 5-fold cross-validation grouped by client rather than by individual page. A model trained on some of a client's pages and tested on others would leak that client's own patterns across the split. Every score used for evaluation is out of fold, meaning no page is ever scored by a model that trained on it or on another page from the same client.

**Cross-fold calibration:** Validation trains five separate models, one per fold, and their out-of-fold scores get pooled into a single column for ranking. That only works if all five models' scores mean the same thing. They didn't. One fold's model produced higher raw scores without being more accurate, and a direct check confirmed it: a naive pooled ranking gave that one fold 93 to 100 percent of the top of the queue at every depth checked, while the fold with the most real positives was almost entirely absent from it.

Two fixes were tried first and rejected. Fitting one shared probability curve across all folds could not tell which fold a row came from, so it could not correct two folds that needed opposite corrections. Fitting a separate curve per fold fixed that part, but each curve was estimated from a limited sample and came out a different shape, so the same imbalance reappeared with a different fold in the lead.

Eventually a simpler fix that held up was decided upon: converting each fold's scores to a percentile rank within that fold. Nothing gets estimated, so nothing can come out uneven between folds by chance.

**Queue ranking:** The queue was originally meant to rank by the model's score with a small nudge from the rule, so a close call could still be moved by a known signal. Once the calibration fix was in place, that combination was tested several different ways: the original nudge at its original weight, several smaller weights, sorting by rule tier first, and a percentile blend of the two scores. Every version reintroduced some version of the same fold imbalance the calibration fix was meant to remove. The queue ranks on the calibrated model score alone. The rule still decides every page's archetype and recommended action, it just no longer has any influence on sort order.

**Leakage checks:** Four checks were run before any result was trusted. The first confirmed that no feature's time window crosses into the label's window. The second confirmed that the eligibility filter used to build the label doesn't itself encode the outcome. The third confirmed that no feature is a near-duplicate of the label under a different name. The fourth was a deliberate injection check, where a known-leaky column was added on purpose to confirm the validation setup would actually catch it rather than assuming a clean-looking score always means a clean pipeline. All four passed.

In [5]:
import sys
sys.path.insert(0, "work/scripts")
from w07_pipeline_utils import FEATURE_COLS, RANDOM_STATE, N_FOLDS

print("Features:", FEATURE_COLS)
print(f"GroupKFold folds: {N_FOLDS}, grouped by client_hash_id")
print(f"Random Forest: n_estimators=300, max_depth=6, min_samples_leaf=20, random_state={RANDOM_STATE}")
print(f"Baseline rule: baseline_score = zero_clicks_at_position*2 + position_worsened")
print(f"Calibration: percentile rank within each fold (oof_rf_score_calibrated)")
print(f"Queue ranking: oof_rf_score_calibrated alone, no rule-based combination")

Features: ['avg_position_fh', 'log_impressions_fh', 'log_clicks_fh', 'ctr_fh', 'position_change', 'has_position_trend']
GroupKFold folds: 5, grouped by client_hash_id
Random Forest: n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42
Baseline rule: baseline_score = zero_clicks_at_position*2 + position_worsened
Calibration: percentile rank within each fold (oof_rf_score_calibrated)
Queue ranking: oof_rf_score_calibrated alone, no rule-based combination


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Precision at K, mean and standard deviation across the 5 held-out folds. The model's score is the calibrated version described in Methodology. Base rate, the share of pages actually labeled declining, is shown alongside it since a high score means little without knowing what a lucky guess would already get.

| K | baseline_rule | model (calibrated) | base rate |
|---|---|---|---|
| 20 | 0.490 ± 0.124 | 0.530 ± 0.057 | 0.438 |
| 50 | 0.420 ± 0.107 | 0.556 ± 0.065 | 0.438 |
| 100 | 0.412 ± 0.119 | 0.582 ± 0.075 | 0.438 |
| 200 | 0.433 ± 0.131 | 0.578 ± 0.056 | 0.438 |

At K=50, the depth this work is framed around, the model beats the rule in every fold: 0.66 vs 0.48, 0.58 vs 0.54, 0.50 vs 0.28, 0.52 vs 0.46, 0.52 vs 0.34. Five wins out of five, not an average hiding a mixed result.

At K=100 and K=200 the gap holds under a bootstrap check on the fold-level differences. The 95 percent interval sits at [0.082, 0.240] at K=100 and [0.045, 0.212] at K=200, both entirely above zero.

**Fold fairness.** However, there is a second, separate question. Does the ranking draw evenly from every fold, or does one fold's rows crowd out the rest regardless of which pages are actually declining? Before calibration, the answer was no. Pooling five independently trained models' raw scores gave one single fold 93 to 100 percent of the top of the queue at every depth checked, while the fold with the highest real base rate was almost entirely absent from it. After calibration, every fold contributes exactly 20 percent of the top of the queue at every depth checked, matching what 5 evenly sized folds should produce if none of them were being favored by an artifact of how the scores happened to be pooled.

In [6]:
import json

with open("work/outputs/capstone_precision_at_k.json") as f:
    precision = json.load(f)
with open("work/outputs/fold_representation_check.json") as f:
    fold_check = json.load(f)

print(f"base_rate: {precision['base_rate']:.3f}\n")

print(f"{'K':<6}{'baseline_rule':<20}{'model (calibrated)':<22}")
for k in ["20", "50", "100", "200"]:
    b = precision["summary_mean_std"][k]["baseline_rule"]
    m = precision["summary_mean_std"][k]["oof_rf_score_calibrated"]
    print(f"{k:<6}{b['mean']:.3f} +/- {b['std']:.3f}{'':<6}{m['mean']:.3f} +/- {m['std']:.3f}")

print("\nK=50, fold by fold:")
k50 = [r for r in precision["fold_level"] if r["k"] == 50]
wins = 0
for r in sorted(k50, key=lambda r: r["fold"]):
    won = r["oof_rf_score_calibrated"] > r["baseline_rule"]
    wins += won
    print(f"  fold {r['fold']}: baseline={r['baseline_rule']:.2f}  model={r['oof_rf_score_calibrated']:.2f}  {'model wins' if won else 'baseline wins'}")
print(f"model wins {wins}/5 folds")

print("\nBootstrap 95% CI, model - baseline:")
for k in ["50", "100", "200"]:
    ci = precision["bootstrap_ci"][k]["calibrated_vs_baseline"]
    print(f"  K={k}: mean_diff={ci['mean_diff']:.3f}  CI=[{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]")

print("\nFold representation of the top-K queue (share per fold, calibrated score):")
for k in ["20", "50", "100", "200"]:
    shares = fold_check["top_k_representation"]["oof_rf_score_calibrated"][k]["share_by_fold"]
    print(f"  K={k}: " + ", ".join(f"fold{f}={s:.0%}" for f, s in shares.items()))

base_rate: 0.438

K     baseline_rule       model (calibrated)    
20    0.490 +/- 0.124      0.530 +/- 0.057
50    0.420 +/- 0.107      0.556 +/- 0.065
100   0.412 +/- 0.119      0.582 +/- 0.075
200   0.433 +/- 0.131      0.578 +/- 0.056

K=50, fold by fold:
  fold 1: baseline=0.48  model=0.66  model wins
  fold 2: baseline=0.54  model=0.58  model wins
  fold 3: baseline=0.28  model=0.50  model wins
  fold 4: baseline=0.46  model=0.52  model wins
  fold 5: baseline=0.34  model=0.52  model wins
model wins 5/5 folds

Bootstrap 95% CI, model - baseline:
  K=50: mean_diff=0.136  CI=[0.072, 0.196]
  K=100: mean_diff=0.170  CI=[0.082, 0.240]
  K=200: mean_diff=0.145  CI=[0.045, 0.212]

Fold representation of the top-K queue (share per fold, calibrated score):
  K=20: fold1=20%, fold2=20%, fold3=20%, fold4=20%, fold5=20%
  K=50: fold1=20%, fold2=20%, fold3=20%, fold4=20%, fold5=20%
  K=100: fold1=20%, fold2=20%, fold3=20%, fold4=20%, fold5=20%
  K=200: fold1=20%, fold2=20%, fold3=20%, fold4=

## 5. Limitations

*What this work cannot claim.*

**About a fifth of the population has no rule signal at all.** 31,499 of 150,675 pages, roughly 21 percent, have no valid week 1 versus week 2 position data and are treated as non-worsening by default. Of those, 87.8 percent get no rule-based flag at all and depend entirely on the model to be noticed. The remaining 12.2 percent are flagged through zero-clicks but capped at a less urgent action, since the more urgent combined flag needs position data they don't have. This was found during later validation, not during the original build.

**One month, not a trend.** Everything here comes from a single month. The label is a within-month comparison, first half against second half. Nothing has been checked against a different month, a different season, or a longer window.

**The review capacity is assumed.** K=50 was chosen because it's where the model's advantage over the rule held up most consistently in validation. No real reviewer capacity was specified. A different real capacity could change which recommendation applies.

**Equal fold representation is enforced by the calibration method, and the folds aren't actually equal.** The folds have real base rates ranging from 0.33 to 0.56. A fold with more true positives could reasonably contribute more than an equal share to the top of the ranking, and calibration doesn't allow that; every fold gets exactly 20 percent. A version that scaled each fold's contribution by its base rate was tested and made things worse, one fold took the entire top of the queue, so equal representation was kept. The tradeoff: a severe, clearly wrong bias got fixed, at the cost of not reflecting a real difference between folds.

**The calibrated score ranks pages correctly, but the number itself no longer means "probability of decline."** Within each fold, calibration is just a percentile transform of that fold's raw score, so it can't change which pages rank above which. The K=50 result in Results, beating the rule in 5 of 5 folds, is this exact score doing that. What's lost is the raw model's actual value: 0.73 used to approximate a 73 percent chance of decline. The calibrated 0.73 only means a page ranks better than about 73 percent of pages scored by the same fold's model. It can sort a queue but it can't support anything that needs the real magnitude, like an expected-value calculation.

**The calibration also only has meaning within a fold that exists.** Every page in this dataset has a `fold_id` from cross-validation, and calibration is computed relative to that fold's own score distribution. A new page scored next month wouldn't belong to any fold at all, since cross-validation doesn't run at scoring time in production. This validates that the ranking approach works. It is not, as built, a live monthly tool. Retraining on a different historical month would work close to as-is, once the hardcoded date boundaries are parameterized.

Scoring genuinely new pages going forward is the real gap: it would need a persisted model instead of five temporary fold models, and a calibration mapping built to generalize to new scores instead of one that only makes sense within an existing fold. A plausible cadence exists, score mid-month using the last fully trained model, retrain once a month completes and its label exists, but it hasn't been built or tested. Retraining on a schedule like that would keep the model from going stale, but it wouldn't by itself catch a broken pipeline or a real drop in accuracy; that needs its own tracking, which also doesn't exist yet.

**The calibration method was picked by comparing several options against the same five folds.** Cross-validation protects each individual method's own reported number from leakage, but choosing the best of several methods by their results on the same data is a mild form of selection bias. The numbers reported are for the method that was chosen this way, not for one method fixed in advance and checked once.

**Two gaps in what the model can see.** A direct zero-click signal was built but left out of the features; it only backed an early diagnostic. The model sees click-through problems indirectly, through CTR and click volume. There is also no validated signal anywhere in this pipeline for pages that get clicks but bounce immediately. The only candidate for that pattern, tested early on, was confounded with impression volume.

**Confidence tiers were tuned by hand.** The thresholds behind the high and low confidence split were set manually and were never checked against whether low-confidence rows are actually less reliable than high-confidence ones.

In [7]:
no_trend = model_df[model_df["has_position_trend"] == 0]
share = len(no_trend) / len(model_df)
print(f"No trend data: {len(no_trend):,} / {len(model_df):,} rows ({share:.1%})")

breakdown = no_trend["baseline_score"].value_counts(normalize=True).sort_index()
print("\nOf those, share by resulting baseline_score:")
print(breakdown)

no_signal_share = (no_trend["baseline_score"] == 0).mean()
print(f"\nShare with no rule-based signal at all (baseline_score == 0): {no_signal_share:.1%}")

print(f"\nFold base rates (for the equal-representation tradeoff above):")
print(model_df.groupby("fold_id")["is_declining_proxy"].mean().round(3))

No trend data: 31,499 / 150,675 rows (20.9%)

Of those, share by resulting baseline_score:
baseline_score
0    0.877552
2    0.122448
Name: proportion, dtype: float64

Share with no rule-based signal at all (baseline_score == 0): 87.8%

Fold base rates (for the equal-representation tradeoff above):
fold_id
1    0.436
2    0.478
3    0.391
4    0.560
5    0.328
Name: is_declining_proxy, dtype: float64


In [10]:
process = subprocess.Popen(
    [sys.executable, "-u", "work/scripts/02_build_queue.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
process.wait()
assert process.returncode == 0, f"02_build_queue.py failed with exit code {process.returncode}"

  Assigning archetypes...
  Assigning confidence...
  Sorting ranked queue...
Archetype mix:
archetype
no_flag                     65570
position_worsened_only      52753
zero_clicks_only            14457
zero_clicks_and_worsened    10609
model_only_catch             7286
Name: count, dtype: int64
Wrote /content/FlyRank-ML-Track/work/outputs/w07_metrics.json
Wrote /content/FlyRank-ML-Track/work/outputs/w07_report.md
Wrote charts to /content/FlyRank-ML-Track/work/outputs/charts
Wrote /content/FlyRank-ML-Track/work/data/processed/w07_ranked_queue.csv (local only — gitignored, not committed)


In [11]:
queue_df = pd.read_csv("work/data/processed/w07_ranked_queue.csv")

outcome_by_confidence = (
    queue_df.groupby(["archetype", "confidence"])["is_declining_proxy"]
    .agg(["mean", "count"])
    .round(3)
)
print(outcome_by_confidence)

                                      mean  count
archetype                confidence              
model_only_catch         high        0.449   1884
                         low         0.536   5402
no_flag                  high        0.331  44227
                         low         0.579  21343
position_worsened_only   high        0.436  48348
                         low         0.469   4405
zero_clicks_and_worsened high        0.529   7876
                         low         0.606   2733
zero_clicks_only         high        0.421   9646
                         low         0.472   4811


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
